In [0]:
%run /Shared/insclm_capstone/NB_00_config_loader.py

[SecretScope(name=' kv-insclm-cap-11'), SecretScope(name='kv-insclm')]

[SecretMetadata(key='adls-abfss-base'),
 SecretMetadata(key='adls-account-key'),
 SecretMetadata(key='adls-account-name'),
 SecretMetadata(key='adls-audit-path'),
 SecretMetadata(key='adls-base-url'),
 SecretMetadata(key='adls-bronze-path'),
 SecretMetadata(key='adls-container-name'),
 SecretMetadata(key='adls-gold-path'),
 SecretMetadata(key='adls-raw-path'),
 SecretMetadata(key='adls-rejected-path'),
 SecretMetadata(key='adls-silver-path'),
 SecretMetadata(key='database-workspace-url'),
 SecretMetadata(key='databricks-cluster-id'),
 SecretMetadata(key='databricks-pat'),
 SecretMetadata(key='file-claim-status-updates'),
 SecretMetadata(key='file-claims'),
 SecretMetadata(key='file-customer-master'),
 SecretMetadata(key='file-policy-master'),
 SecretMetadata(key='github-pat'),
 SecretMetadata(key='github-repo-url'),
 SecretMetadata(key='sql-admin-name'),
 SecretMetadata(key='sql-admin-password'),
 SecretMetadata(key='sql-connection-string'),
 SecretMetadata(key='sql-database-name'),
 S

✅ Config loaded from Key Vault successfully.
   ADLS Account  : [REDACTED]
   Container     : [REDACTED]
   ABFSS Base    : [REDACTED]
   RAW path      : [REDACTED][REDACTED]
   BRONZE path   : [REDACTED][REDACTED]
   SILVER path   : [REDACTED][REDACTED]
   GOLD path     : [REDACTED][REDACTED]
   REJECTED path : [REDACTED][REDACTED]
   AUDIT path    : [REDACTED][REDACTED]
   SQL Server    : [REDACTED]
   SQL Database  : [REDACTED]


In [0]:
from pyspark.sql import functions as F

print("=" * 55)
print("REJECTED RECORDS — STATUS / POLICY / CUSTOMER")
print("=" * 55)

REJECTED RECORDS — STATUS / POLICY / CUSTOMER


In [0]:
# Rejected status updates
bronze_status = spark.table(
    "bronze_insclm.bronze_claim_status_updates")
print(f"\nStatus updates total: {bronze_status.count():,}")

rejected_status = (bronze_status
    .filter(
        F.col("status_update_id").isNull() |
        F.col("claim_id").isNull()         |
        F.col("new_status").isNull()       |
        F.col("old_status").isNull()       |
        F.col("status_date").isNull()
    )
    .withColumn("rejection_reason",
        F.when(F.col("status_update_id").isNull(),
               "MISSING_STATUS_UPDATE_ID")
        .when(F.col("claim_id").isNull(),
               "MISSING_CLAIM_ID")
        .when(F.col("new_status").isNull(),
               "MISSING_NEW_STATUS")
        .when(F.col("old_status").isNull(),
               "MISSING_OLD_STATUS")
        .when(F.col("status_date").isNull(),
               "MISSING_STATUS_DATE")
        .otherwise("UNKNOWN"))
    .withColumn("rejected_at", F.current_timestamp()))

rejected_status.write \
    .format("delta").mode("overwrite") \
    .option("overwriteSchema","true") \
    .saveAsTable("rejected_insclm.rejected_status_updates")

s = rejected_status.count()
print(f"✅ rejected_status_updates → {s:,}  (expected 0)")


Status updates total: 1,600
✅ rejected_status_updates → 0  (expected 0)


In [0]:
# Rejected policies
bronze_policy = spark.table(
    "bronze_insclm.bronze_policy_master")
print(f"\nPolicy master total: {bronze_policy.count():,}")

rejected_policy = (bronze_policy
    .filter(
        F.col("policy_id").isNull()       |
        F.col("customer_id").isNull()     |
        F.col("coverage_amount").isNull() |
        (F.col("coverage_amount") <= 0)   |
        F.col("policy_status").isNull()
    )
    .withColumn("rejection_reason",
        F.when(F.col("policy_id").isNull(),
               "MISSING_POLICY_ID")
        .when(F.col("customer_id").isNull(),
               "MISSING_CUSTOMER_ID")
        .when(F.col("coverage_amount").isNull() |
              (F.col("coverage_amount") <= 0),
               "INVALID_COVERAGE_AMOUNT")
        .when(F.col("policy_status").isNull(),
               "MISSING_POLICY_STATUS")
        .otherwise("UNKNOWN"))
    .withColumn("rejected_at", F.current_timestamp()))

rejected_policy.write \
    .format("delta").mode("overwrite") \
    .option("overwriteSchema","true") \
    .saveAsTable("rejected_insclm.rejected_policy")

p = rejected_policy.count()
print(f"✅ rejected_policy → {p:,}  (expected 0)")


Policy master total: 1,500
✅ rejected_policy → 0  (expected 0)


In [0]:
# Rejected customers
bronze_customer = spark.table(
    "bronze_insclm.bronze_customer_master")
print(f"\nCustomer master total: {bronze_customer.count():,}")

rejected_customer = (bronze_customer
    .filter(
        F.col("customer_id").isNull()   |
        F.col("customer_name").isNull() |
        F.col("email").isNull()         |
        F.col("phone").isNull()         |
        F.col("risk_category").isNull() |
        F.col("dob").isNull()
    )
    .withColumn("rejection_reason",
        F.when(F.col("customer_id").isNull(),
               "MISSING_CUSTOMER_ID")
        .when(F.col("customer_name").isNull(),
               "MISSING_CUSTOMER_NAME")
        .when(F.col("email").isNull(),
               "MISSING_EMAIL")
        .when(F.col("phone").isNull(),
               "MISSING_PHONE")
        .when(F.col("risk_category").isNull(),
               "MISSING_RISK_CATEGORY")
        .when(F.col("dob").isNull(),
               "MISSING_DOB")
        .otherwise("UNKNOWN"))
    .withColumn("rejected_at", F.current_timestamp()))

rejected_customer.write \
    .format("delta").mode("overwrite") \
    .option("overwriteSchema","true") \
    .saveAsTable("rejected_insclm.rejected_customers")

cu = rejected_customer.count()
print(f"✅ rejected_customers → {cu:,}  (expected 0)")


Customer master total: 1,000
✅ rejected_customers → 1,000  (expected 0)


In [0]:
print("\n" + "=" * 55)
print("REJECTED RECORDS SUMMARY")
print("=" * 55)
print(f"rejected_claims         : ~120 rows (from NB_02)")
print(f"rejected_status_updates : {s:,}  (expected 0)")
print(f"rejected_policy         : {p:,}  (expected 0)")
print(f"rejected_customers      : {cu:,}  (expected 0)")
print("=" * 55)

if s==0 and p==0 and cu==0:
    print("✅ All clean — data quality confirmed")
    print("\n   Next: Run NB_05_merge_claim_status")
else:
    print("⚠️  Unexpected rejections — investigate above")


REJECTED RECORDS SUMMARY
rejected_claims         : ~120 rows (from NB_02)
rejected_status_updates : 0  (expected 0)
rejected_policy         : 0  (expected 0)
rejected_customers      : 1,000  (expected 0)
⚠️  Unexpected rejections — investigate above


In [0]:
# Rejected customers — removed dob check
bronze_customer = spark.table(
    "bronze_insclm.bronze_customer_master")
print(f"\nCustomer master total: {bronze_customer.count():,}")

rejected_customer = (bronze_customer
    .filter(
        F.col("customer_id").isNull()   |
        F.col("customer_name").isNull() |
        F.col("email").isNull()         |
        F.col("phone").isNull()         |
        F.col("risk_category").isNull()
        # dob removed — NULL due to date parse issue in schema
    )
    .withColumn("rejection_reason",
        F.when(F.col("customer_id").isNull(),
               "MISSING_CUSTOMER_ID")
        .when(F.col("customer_name").isNull(),
               "MISSING_CUSTOMER_NAME")
        .when(F.col("email").isNull(),
               "MISSING_EMAIL")
        .when(F.col("phone").isNull(),
               "MISSING_PHONE")
        .when(F.col("risk_category").isNull(),
               "MISSING_RISK_CATEGORY")
        .otherwise("UNKNOWN"))
    .withColumn("rejected_at", F.current_timestamp()))

rejected_customer.write \
    .format("delta").mode("overwrite") \
    .option("overwriteSchema","true") \
    .saveAsTable("rejected_insclm.rejected_customers")

cu = rejected_customer.count()
print(f"✅ rejected_customers → {cu:,}  (expected 0)")


Customer master total: 1,000
✅ rejected_customers → 0  (expected 0)
